In [ ]:
# 1. Clone repository & install ONLY missing helper packages
!git clone https://github.com/ahsan-c0ding/S4-Enhancement-Exploration.git
%cd S4-Enhancement-Exploration
!git checkout python

import os
import sys
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

# FIXED: Only install lightweight missing packages (Do NOT reinstall torch/torchvision)
%pip install einops coloredlogs torchinfo --quiet
%pip install git+https://github.com/mwalmsley/galaxy_mnist.git@c1fe9853a00bc34b2ff082585c6bb1654d34d239 --quiet

# Symlink workaround for model_params path resolving
parent_dir = os.path.dirname(current_dir)
_shim_path = os.path.join(parent_dir, "model_params")
_real_path = os.path.join(current_dir, "model_params")
if not os.path.exists(_shim_path) and os.path.exists(_real_path):
    os.symlink(_real_path, _shim_path)

In [ ]:
import math
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from einops import repeat

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_score, recall_score, roc_auc_score

from model.functions import load_data

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")
plt.rcParams["figure.figsize"] = [11, 6]

# ============================================================================
# SEED -- deliberately DIFFERENT from the rest of the series (30485).
# This is a genuine repeat: different weight init, different train/val split,
# different minibatch shuffling, different augmentation draws. The test set
# itself is unaffected -- load_data returns the same fixed test split
# regardless of seed, so the final comparison is still apples-to-apples.
# ============================================================================
RNG_SEED = 8842
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(RNG_SEED)

CLASS_NAMES = ["Smooth Round", "Smooth Cigar", "Edge-on Disk", "Unbarred Spiral"]

# ============================================================================
# PRODUCTION S4D CONFIGURATION (reference values)
# ============================================================================
COLORED = True
IN_CHANNELS = 3 if COLORED else 1

S4D_STATE = 256
S4D_NUM_LAYERS = 3
S4D_PATCH_SIZE = 4
S4D_POOLING = "last"
S4D_USE_NORM = False
S4D_USE_RESIDUAL = False
S4D_DROPOUT = 0.2
S4D_PATCH_EMBED = "conv"

S4D_BATCH_SIZE = 32
S4D_FINAL_EPOCHS = 630
LEARNING_RATE = 1e-3

# ============================================================================
# REPEAT RUN: LINEAR+S4D, d~108, SEED 2
# ============================================================================
# The original d=108 run (seed 30485) scored 78.95% test accuracy, breaking
# the otherwise-flat 72->90 trend (80.20% / 80.10%). This repeats that exact
# config under a different seed to check whether 78.95% reproduces (real
# effect) or was a one-off low draw (noise).
REPEAT_D = 108
REPEAT_LAYERS = S4D_NUM_LAYERS   # 3, matches the original d=108 run
REPEAT_POOLING = S4D_POOLING     # "last", matches the original d=108 run
REPEAT_PATCH_EMBED = "linear"

print(f"Device: {DEVICE}")
print(f"Seed: {RNG_SEED} (original d=108 run used 30485)")
print(f"Repeat run -- Linear+S4D: d={REPEAT_D} | layers={REPEAT_LAYERS} | pooling={REPEAT_POOLING} | embed=linear")


In [ ]:
class HilbertScan(nn.Module):
    """Reorders patches of a (B, C, H, W) image along a Hilbert curve."""
    def __init__(self, image_size=64, patch_size=1):
        super().__init__()
        assert image_size % patch_size == 0, "image_size must be divisible by patch_size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.register_buffer("indices", self._get_hilbert_indices(self.grid_size))

    @staticmethod
    def _rot(s, x, y, rx, ry):
        if ry == 0:
            if rx == 1:
                x = s - 1 - x
                y = s - 1 - y
            x, y = y, x
        return x, y

    def _d2xy(self, n, d):
        x = y = 0
        t, s = d, 1
        while s < n:
            rx = (t // 2) & 1
            ry = (t ^ rx) & 1
            x, y = self._rot(s, x, y, rx, ry)
            x += s * rx
            y += s * ry
            t //= 4
            s *= 2
        return x, y

    def _get_hilbert_indices(self, grid_size):
        indices = []
        for d in range(grid_size * grid_size):
            x, y = self._d2xy(grid_size, d)
            indices.append(y * grid_size + x)
        return torch.LongTensor(indices)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        patches = x.unfold(2, p, p).unfold(3, p, p)
        patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        patches = patches.view(B, self.num_patches, C * p * p)
        return patches[:, self.indices, :]


class TakeLastTimestep(nn.Module):
    def forward(self, x):
        return x[:, -1, :]


class S4DConv(nn.Module):
    """Fast FFT-based parallel convolution S4D layer."""
    def __init__(self, d_model, d_state=64, dt_min=0.001, dt_max=0.1, transposed=True, lr=None):
        super().__init__()
        self.h = d_model
        self.n = d_state
        self.transposed = transposed

        log_dt = torch.rand(self.h) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)
        log_A_real = torch.log(0.5 * torch.ones(self.h, self.n // 2))
        A_imag = math.pi * repeat(torch.arange(self.n // 2), 'n -> h n', h=self.h)
        C_init = torch.randn(self.h, self.n // 2, dtype=torch.cfloat)

        self.register("log_dt", log_dt, lr)
        self.register("log_A_real", log_A_real, lr)
        self.register("A_imag", A_imag, lr)

        self.C = nn.Parameter(torch.view_as_real(C_init))
        self.D = nn.Parameter(torch.randn(self.h))

    def register(self, name, tensor, lr=None):
        if lr == 0.0:
            self.register_buffer(name, tensor)
        else:
            self.register_parameter(name, nn.Parameter(tensor))
            optim = {"weight_decay": 0.0}
            if lr is not None:
                optim["lr"] = lr
            setattr(getattr(self, name), "_optim", optim)

    def forward(self, u):
        if not self.transposed:
            u = u.transpose(-1, -2)
        L = u.size(-1)

        dt = torch.exp(self.log_dt)
        C = torch.view_as_complex(self.C)
        A = -torch.exp(self.log_A_real) + 1j * self.A_imag

        dtA = A * dt.unsqueeze(-1)
        K_exp = torch.exp(dtA.unsqueeze(-1) * torch.arange(L, device=u.device))
        C_tilde = C * (torch.exp(dtA) - 1.) / A
        k = 2 * torch.einsum('hn, hnl -> hl', C_tilde, K_exp).real

        k_f = torch.fft.rfft(k, n=2 * L)
        u_f = torch.fft.rfft(u, n=2 * L)
        y = torch.fft.irfft(u_f * k_f, n=2 * L)[..., :L]
        y = y + u * self.D.unsqueeze(-1)

        if not self.transposed:
            y = y.transpose(-1, -2)
        return y, None


class ConvPatchStem(nn.Module):
    """Convolutional stem for local neighborhood mixing before patch projection."""
    def __init__(self, in_channels, d_model, patch_size):
        super().__init__()
        mid_channels = max(in_channels * 8, 32)
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, stride=1, padding=1),
            nn.GELU(),
            nn.Conv2d(mid_channels, d_model, kernel_size=patch_size, stride=patch_size),
        )

    def forward(self, x):
        return self.net(x)


class GalaxyClassifierS4DFast(nn.Module):
    """Production S4D Classifier."""
    def __init__(self, s4_state=64, d_model=64, num_classes=4, colored=True,
                 num_layers=2, patch_size=1, pooling="last",
                 use_norm=False, use_residual=False, dropout=0.0,
                 patch_embed="linear"):
        super().__init__()
        self.hilbert_channels = 1 if not colored else 3
        self.patch_size = patch_size
        self.pooling = pooling
        self.use_norm = use_norm
        self.use_residual = use_residual
        self.patch_embed = patch_embed

        if patch_embed == "linear":
            self.hilbert_scan = HilbertScan(image_size=64, patch_size=patch_size)
            patch_dim = self.hilbert_channels * patch_size * patch_size
            self.uproject = nn.Linear(patch_dim, d_model)
            self.conv_stem = None
        elif patch_embed == "conv":
            self.conv_stem = ConvPatchStem(self.hilbert_channels, d_model, patch_size)
            self.hilbert_scan = HilbertScan(image_size=64 // patch_size, patch_size=1)
            self.uproject = nn.Identity()

        self.s4_layers = nn.ModuleList([
            S4DConv(d_model=d_model, d_state=s4_state, transposed=False)
            for _ in range(num_layers)
        ])
        self.acts = nn.ModuleList([nn.GELU() for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(num_layers)]) if use_norm else None
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

        if pooling == "last":
            self.take_last = TakeLastTimestep()
        elif pooling == "mean":
            self.take_last = None
        else:
            raise ValueError(f"Unknown pooling type {pooling}")

        self.fc = nn.Linear(d_model, num_classes)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=True):
        if self.patch_embed == "conv":
            feat = self.conv_stem(x)
            x_seq = self.hilbert_scan(feat)
            h = self.uproject(x_seq)
        else:
            x_seq = self.hilbert_scan(x)
            h = self.uproject(x_seq)

        for i, (s4_layer, act) in enumerate(zip(self.s4_layers, self.acts)):
            residual = h
            h_in = self.norms[i](h) if self.use_norm else h
            h_out, _ = s4_layer(h_in)
            h_out = act(h_out)
            h_out = self.drop(h_out)
            h = residual + h_out if self.use_residual else h_out

        pooled = h.mean(dim=1) if self.take_last is None else self.take_last(h)
        logits = self.fc(pooled)

        if return_logits:
            return logits
        return self.softmax(logits)

In [ ]:
# Load dataset
X, y_onehot, y = load_data(root="./data", download=True, train=True, colored=COLORED)
X_test, y_test_onehot, y_test = load_data(root="./data", download=True, train=False, colored=COLORED)
NUM_CLASSES = y_onehot.shape[1]

# FIXED: Unpack 4 outputs corresponding to passing X and y
x_train, x_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RNG_SEED, stratify=y
)

# Custom dataset supporting rotation and flip augmentation
class GalaxyDataset(Dataset):
    def __init__(self, images, labels, augment=False):
        self.images = images
        self.labels = labels
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        lbl = self.labels[idx]
        if self.augment:
            # Random orthogonal rotation
            k = random.randint(0, 3)
            img = torch.rot90(img, k, [1, 2])
            # Random flips
            if random.random() > 0.5:
                img = torch.flip(img, [2])
            if random.random() > 0.5:
                img = torch.flip(img, [1])
        return img, lbl

train_dataset = GalaxyDataset(x_train, y_train, augment=True)
val_dataset = GalaxyDataset(x_val, y_val, augment=False)
test_dataset = GalaxyDataset(X_test, y_test, augment=False)

train_loader = DataLoader(train_dataset, batch_size=S4D_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=S4D_BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=S4D_BATCH_SIZE, shuffle=False)

print(f"Data Split -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
def create_s4d_optimizer(model, lr=1e-3, weight_decay=0.01):
    decay_params, no_decay_params, special_params = [], [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if hasattr(param, "_optim"):
            opt_dict = param._optim
            param_lr = opt_dict.get("lr", lr)
            special_params.append({'params': [param], 'lr': param_lr, 'weight_decay': 0.0})
        elif any(k in name for k in ["bias", "norm", "LayerNorm"]):
            no_decay_params.append(param)
        else:
            decay_params.append(param)

    groups = [
        {'params': decay_params, 'weight_decay': weight_decay, 'lr': lr},
        {'params': no_decay_params, 'weight_decay': 0.0, 'lr': lr},
    ] + special_params
    return torch.optim.AdamW(groups)


def train_s4d_production(model, train_loader, val_loader, epochs=S4D_FINAL_EPOCHS, lr=LEARNING_RATE, device=DEVICE, ckpt_name="s4d_production_best.pt"):
    model.to(device)
    optimizer = create_s4d_optimizer(model, lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-5
    )
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "train_acc": [], "val_acc": []}
    best_val_acc = 0.0
    os.makedirs("checkpoints", exist_ok=True)
    best_ckpt_path = f"checkpoints/{ckpt_name}"  # parametrized so different runs don't clobber each other's checkpoint

    start_time = time.time()
    print(f"Starting S4D production training for {epochs} epochs...")

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(images, return_logits=True)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(-1) == labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        train_loss = running_loss / total
        train_acc = correct / total

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                logits = model(images, return_logits=True)
                val_correct += (logits.argmax(-1) == labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_ckpt_path)

        if (epoch + 1) % 25 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch+1:03d}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} (Best: {best_val_acc:.4f})")

    elapsed = time.time() - start_time
    print(f"\nTraining completed in {elapsed/60:.2f} mins. Best Validation Accuracy: {best_val_acc*100:.2f}%")
    
    # Load best checkpoint
    model.load_state_dict(torch.load(best_ckpt_path))
    return model, history


def evaluate_on_test(model, test_loader, device=DEVICE):
    """Runs a trained model on the test set and returns every metric we report."""
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            probs = model(images, return_logits=False)
            preds = probs.argmax(-1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(labels.numpy())
            all_probs.extend(probs.cpu().numpy())
    all_probs = np.array(all_probs)
    return {
        "accuracy": accuracy_score(all_targets, all_preds),
        "f1": f1_score(all_targets, all_preds, average="macro"),
        "precision": precision_score(all_targets, all_preds, average="macro"),
        "recall": recall_score(all_targets, all_preds, average="macro"),
        "auc": roc_auc_score(all_targets, all_probs, multi_class="ovr", average="macro"),
        "preds": all_preds,
        "targets": all_targets,
    }


In [ ]:
# Repeat run: linear embed, d=108, 3 layers, S4D on, last pooling -- same
# config as the original d=108 run, different seed (see cell above).
repeat_d108_model = GalaxyClassifierS4DFast(
    s4_state=REPEAT_D,
    d_model=REPEAT_D,
    num_classes=NUM_CLASSES,
    colored=COLORED,
    num_layers=REPEAT_LAYERS,
    patch_size=S4D_PATCH_SIZE,
    pooling=REPEAT_POOLING,
    use_norm=S4D_USE_NORM,
    use_residual=S4D_USE_RESIDUAL,
    dropout=S4D_DROPOUT,
    patch_embed=REPEAT_PATCH_EMBED
)

best_repeat_d108_model, repeat_d108_history = train_s4d_production(
    repeat_d108_model, train_loader, val_loader,
    epochs=S4D_FINAL_EPOCHS, ckpt_name="linear_d108_seed2_best.pt"
)


In [ ]:
repeat_res = evaluate_on_test(best_repeat_d108_model, test_loader)

print("=" * 50)
print("Linear+S4D d=108 (seed 2) -- Test Set Results")
print("=" * 50)
print(f"Accuracy:        {repeat_res['accuracy'] * 100:.2f}%")
print(f"Macro-F1:        {repeat_res['f1']:.4f}")
print(f"Macro-Precision: {repeat_res['precision']:.4f}")
print(f"Macro-Recall:    {repeat_res['recall']:.4f}")
print(f"Macro-AUC (OVR): {repeat_res['auc']:.4f}")
print("=" * 50)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(repeat_d108_history["train_acc"], label="Train Acc")
axes[0].plot(repeat_d108_history["val_acc"], label="Val Acc")
axes[0].set_title("Linear+S4D d=108 (seed 2) -- Training Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

cm = confusion_matrix(repeat_res["targets"], repeat_res["preds"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Linear+S4D d=108 (seed 2) -- Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()


In [ ]:
from torchinfo import summary

def count_params(model):
    return sum(p.numel() for p in model.parameters())

repeat_params = count_params(best_repeat_d108_model)
print(f"Linear+S4D d=108 (seed 2): {repeat_params:,} params")

print("\nFull layer-by-layer breakdown:")
summary(best_repeat_d108_model, input_size=(S4D_BATCH_SIZE, IN_CHANNELS, 64, 64))


In [ ]:
import shutil
from IPython.display import FileLink, display

ckpt_name = "linear_d108_seed2_best.pt"
source_path = f"checkpoints/{ckpt_name}"
destination_path = f"/kaggle/working/{ckpt_name}"

if os.path.exists(source_path):
    shutil.copy(source_path, destination_path)
    print("✅ Checkpoint moved to the root output folder!")
    print("Click the link below to download your weights:")
    os.chdir('/kaggle/working')
    display(FileLink(ckpt_name))
    os.chdir('/kaggle/working/S4-Enhancement-Exploration')
else:
    print("❌ Could not find the checkpoint -- did training complete successfully?")


In [ ]:
# ============================================================================
# SUMMARY -- does the d=108 dip survive a second seed?
# ============================================================================
original_d108_acc = 0.7895
repeat_d108_acc = repeat_res["accuracy"]

print("=" * 60)
print("d=108 REPEAT CHECK (linear embed, S4D on, 3 layers)")
print("=" * 60)
print(f"Seed 30485 (original): {original_d108_acc*100:.2f}%")
print(f"Seed {RNG_SEED} (repeat):   {repeat_d108_acc*100:.2f}%")
print(f"Spread:                 {abs(original_d108_acc - repeat_d108_acc)*100:.2f}pp")

seed_avg_d108 = (original_d108_acc + repeat_d108_acc) / 2
print(f"\nTwo-seed average at d=108: {seed_avg_d108*100:.2f}%")

print("\n" + "-" * 60)
print("Full width scan for context (d=72 and d=90 are single-seed):")
print("-" * 60)
width_rows = [
    ("d=72",              0.8020),
    ("d=90",              0.8010),
    ("d=108 (seed 30485)", original_d108_acc),
    ("d=108 (seed %d)" % RNG_SEED, repeat_d108_acc),
    ("d=256",             0.8285),
]
for label, acc in width_rows:
    print(f"{label:24s} {acc*100:.2f}%")

spread = abs(original_d108_acc - repeat_d108_acc) * 100
gap_from_90 = 0.8010 - seed_avg_d108
print(f"\nGap between d=90 (80.10%) and the two-seed d=108 average: {gap_from_90*100:.2f}pp")
if spread > gap_from_90 * 100 * 0.5:
    print("The seed-to-seed spread at d=108 is a substantial fraction of the d=90->d=108 gap --")
    print("treat the 'dip' as tentative until more seeds or intermediate widths are run.")
else:
    print("The seed-to-seed spread at d=108 is small relative to the d=90->d=108 gap --")
    print("the dip looks like it survives ordinary run-to-run noise.")
